# 05 - Sugerido de Unidades para Productos Nuevos

**La pregunta:** "voy a lanzar un labial tono Rojo, PVP $23.000, grupo Labial,
tipo Barra. ¿Cuántas unidades pido para la primera compra?"

Este notebook resuelve un problema **distinto** al de los notebooks 01 a 04.

| | Modelo de clustering (nb 01-04) | Modelo de lanzamientos (este nb) |
|---|---|---|
| **Predice desde** | la historia de venta del producto | los atributos del producto |
| **Sirve para** | productos que ya venden | productos que aún no tienen historia |
| **Técnica** | KMeans + factor semestral por cluster | regresión cuantílica + cohorte de análogos |
| **Entrega** | proyección S2 2026 | unidades para los primeros 6 meses de vida |

Un producto a lanzar tiene todas sus variables de comportamiento en cero, así que el
clustering no puede decir nada útil sobre él. Hay que predecir desde lo único que se
conoce antes de vender: **qué es el producto**.

---

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import utils, ingesta
from src import lanzamientos as lz

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
utils.aplicar_estilo_graficas()
%matplotlib inline

print("Raiz del proyecto:", RAIZ)

## 1. El problema del set de entrenamiento

Para aprender "cuánto vende un lanzamiento" hacen falta lanzamientos históricos con
su resultado conocido. De los 1.815 registros producto × canal, **solo 916 sirven**.
El filtro es la decisión más delicada de todo el modelo.

In [ ]:
maestro = ingesta.cargar_maestro()
df = lz.identificar_debut(maestro)
meses = [m for m in cfg.MESES if m in maestro.columns]

print("Distribucion del mes de debut (primer mes con venta):\n")
conteo = df["debut_mes"].fillna("nunca vendio").value_counts()
print(conteo.reindex(["nunca vendio"] + meses).dropna().astype(int).to_string())

### Los cuatro grupos, y por qué solo uno sirve

**385 registros nunca vendieron** → son justamente los que hay que **predecir**.

**352 registros ya vendían en enero de 2024** → hay que **excluirlos**. Enero 2024 es
el primer mes de la ventana de datos, así que un producto que ya vendía ahí pudo haber
lanzado en 2023, en 2020 o en 2015: **su debut real es desconocido**. Esto se llama
*censura a la izquierda*.

Si se tomaran sus primeros 6 meses observados como si fuera un lanzamiento, se estaría
midiendo un producto ya establecido y en régimen, no un arranque. Serían **352
etiquetas equivocadas sobre 1.268** — suficiente para sesgar el modelo hacia arriba de
forma sistemática y hacer que todos los sugeridos salieran inflados.

**162 registros debutaron hace menos de 6 meses** → se excluyen porque su target
todavía está incompleto.

**916 registros con debut limpio y 6 meses observables** → set de entrenamiento.

In [ ]:
ent = lz.construir_entrenamiento(maestro)
ent[["Material", "Descripcion", "Marca", "Canal", "Subcategoria", "Grupo",
     "Tipo", "Tono", "PVP", "debut_mes", "und_primeros_6m"]].head(10)

## 2. El target: unidades en los primeros 6 meses de vida

Se eligió este horizonte porque **es exactamente la decisión que se está tomando**:
cuánto pedir en la primera compra semestral del producto.

In [ ]:
print("Distribucion del target:\n")
print(ent["und_primeros_6m"].describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())

print("\nPor canal:")
print(ent.groupby("Canal")["und_primeros_6m"].agg(
    ["count", "median", "mean", "max"]).round(1).to_string())

### Lectura: el canal es el factor dominante

La mediana de debut es de **709 unidades en Farmatodo, 125 en Exito y 69 en Prosalon**.
Un factor de 10x entre canales.

Esto confirma la decisión de diseño de todo el proyecto: tratar cada producto-canal
como un registro independiente. Pedir la misma cantidad para los tres canales sería un
error de planeación de un orden de magnitud.

La distribución también es **muy sesgada** (mediana 154, media 452, máximo 7.767), así
que el modelo se entrena sobre `log1p` del target.

In [ ]:
modelo_preliminar = lz.ModeloLanzamiento(
    entrenamiento=ent,
    mediana_pvp_subcat=ent.groupby("Subcategoria")["PVP"].median(),
    pvp_global=float(ent["PVP"].median()),
)
fig = lz.graficar_distribucion_target(modelo_preliminar, guardar=False)
plt.show()

## 3. ¿Los atributos realmente predicen algo?

Antes de construir el modelo hay que comprobar que los atributos tienen señal. Si
`Tipo` o `Tono` no diferenciaran nada, lo honesto sería usar la mediana por canal y no
vender un modelo que no aporta.

In [ ]:
fa = ent[ent["Canal"] == "Farmatodo"]

for col in ["Subcategoria", "Tipo", "Grupo"]:
    print(f"\nMediana de unidades en los primeros 6 meses por {col} (Farmatodo):")
    g = fa.groupby(col)["und_primeros_6m"].agg(["count", "median"])
    g = g[g["count"] >= 3].sort_values("median", ascending=False)
    print(g.round(0).to_string())

In [ ]:
# El PVP: mas caro vende menos unidades?
print("Correlacion de Spearman entre PVP y unidades del debut:\n")
for canal, g in ent.groupby("Canal"):
    print(f"  {canal:<12} {g['PVP'].corr(g['und_primeros_6m'], method='spearman'):+.3f}  (n={len(g)})")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (canal, g) in zip(axes, ent.groupby("Canal")):
    ax.scatter(g["PVP"], g["und_primeros_6m"], alpha=0.45, s=25, color="steelblue")
    ax.set_yscale("log")
    ax.set_title(f"{canal}  (n={len(g)})")
    ax.set_xlabel("PVP")
    ax.set_ylabel("Unidades primeros 6 meses (log)")
fig.suptitle("Relacion PVP vs volumen de debut: mas caro, menos unidades",
             fontsize=13, fontweight="bold")
plt.show()

### Lectura

**Sí hay señal.** En Farmatodo la mediana del debut va de 293 unidades (Bálsamo) a
2.382 (Gel) según el tipo, y de 413 (Uñas) a 1.031 (Rostro) según subcategoría.

El **PVP correlaciona negativamente** (−0,23 a −0,32) en los tres canales: productos
más caros venden menos unidades. Es coherente con la intuición comercial y es
información aprovechable.

Un detalle importante: el modelo usa **`pvp_relativo`** (el PVP dividido por la mediana
de PVP de su subcategoría) además del PVP crudo. Un labial de $23.000 no es "caro" en
absoluto — es caro o barato *frente a los otros labiales*. Esa variable relativa
resultó **más informativa que el PVP crudo** (0,074 vs 0,033 de importancia).

## 4. Entrenamiento y validación honesta

### Por qué GroupKFold y no KFold normal

El mismo `Material` aparece en los 3 canales con targets correlacionados. Con un
KFold simple, el registro de Exito puede caer en entrenamiento y el de Farmatodo en
prueba: el modelo "acierta" porque ya vio ese producto, no porque haya aprendido de
los atributos. **La métrica saldría buena y sería mentira.**

Agrupar por `Material` cierra esa fuga. Es la diferencia entre validar y auto-engañarse.

In [ ]:
modelo = lz.entrenar(maestro)

### Lectura de las métricas

Tres métricas porque ninguna sola basta:

- **MAE** (error en unidades): legible, pero lo domina la cola larga.
- **MAE_log** (error en escala logarítmica): refleja el error **relativo**, que es lo
  que importa cuando el target va de 1 a 7.767 unidades.
- **dentro_factor2** (% de casos donde la predicción queda entre la mitad y el doble
  del real): la lectura más honesta para planeación.

El modelo mejora **31%** en MAE_log sobre el baseline de mediana por canal, y el
acierto dentro del factor 2 sube de 44,2% a 63,9%.

**Dicho sin adornos:** el modelo acierta dentro del factor 2 en 2 de cada 3 casos. No
es una bola de cristal. Por eso se entrega un rango P25–P50–P75 y no un número único.

### Calibración de los cuantiles

Un cuantil mal calibrado es peor que no tenerlo: si el "P75" solo cubriera el 55% real,
el comprador creería tener colchón sin tenerlo. Aquí el P75 cubre el **71,8%** real
(objetivo 75%), suficientemente cerca para ser útil.

In [ ]:
modelo.calibracion

## 5. Qué atributos pesan más

In [ ]:
lz.importancia_atributos(modelo, top=15)

In [ ]:
fig = lz.graficar_importancia(modelo, guardar=False)
plt.show()

### Lectura

**El canal domina con el 47% de la importancia.** Todo lo demás se reparte el resto.
Le siguen `pvp_relativo`, el mes de lanzamiento, la marca, y luego los atributos de
producto (subcategoría, grupo, tipo, tono).

El **mes de lanzamiento** pesa más de lo esperado (0,064): lanzar en julio no es igual
que lanzar en enero, porque el segundo semestre vende más — el mismo hallazgo que
apareció en el EDA del notebook 01.

## 6. El caso concreto del negocio

Labial tono Rojo, PVP $23.000, grupo Labial, tipo Barra, en Farmatodo, lanzando en julio.

In [ ]:
r = lz.sugerir_lanzamiento(
    modelo,
    canal="Farmatodo",
    marca="Essence",
    subcategoria="Labios",
    grupo="Labial",
    tipo="Barra",
    tono="Rojo",
    pvp=23000,
    mes_lanzamiento=7,
)

### Cómo leer este resultado

El modelo no entrega solo un número: entrega el número **más la evidencia que lo
respalda**. Eso es deliberado, porque un sugerido sin respaldo no se puede defender en
una mesa de compra.

**La cohorte de análogos** busca en cascada, de lo más específico a lo más general, y se
detiene al reunir 5 comparables:

| # | Criterio |
|---|---|
| 1 | `Canal + Marca + Grupo + Tipo + Tono + PVP ±20%` |
| 2 | `Canal + Marca + Grupo + Tipo + Tono` |
| 3 | `Canal + Marca + Grupo + Tono` |
| 4 | `Canal + Grupo + Tipo + Tono` |
| 5 | `Canal + Grupo + Tono` |
| 6 | `Canal + Marca + Grupo + Tipo` |
| 7 | `Canal + Marca + Grupo` |
| 8 | `Canal + Grupo + Tipo` |
| 9 | `Canal + Grupo` |
| 10 | `Canal + Subcategoría + Tono` |
| 11 | `Canal + Subcategoría + Tipo` |
| 12 | `Canal + Subcategoría` |
| 13 | `Canal` |

**El orden en que se sueltan los atributos está medido, no supuesto.** `Canal` nunca se
suelta (47% de la predicción). `Tono` se suelta tarde porque dentro de un mismo grupo la
mediana de venta varía 2–3x. `Marca` se incluye pero se suelta antes que `Tono`. `Tipo`
se suelta primero, por ser el menos discriminante.

El nivel alcanzado **es la medida de confianza**: llegar a
`Canal + Marca + Grupo + Tipo + Tono` es evidencia fuerte; quedarse en `Canal` a secas
significa que no había nada parecido y el número es apenas una referencia gruesa.

El resultado incluye además `marca_respetada` y `tono_respetado`, para que quede claro
si los comparables que se muestran corresponden de verdad a la marca y el tono
consultados.

**La señal de divergencia** compara el modelo contra la cohorte. Si difieren por más
del doble, el registro se marca: es una combinación de atributos sin respaldo histórico
suficiente y hay que validarla comercialmente antes de pedir.

### Sensibilidad: qué pasa si cambio un atributo

Vale la pena ver cuánto mueve cada decisión, porque eso es lo que el negocio realmente
puede controlar.

In [ ]:
base = dict(canal="Farmatodo", marca="Essence", subcategoria="Labios",
            grupo="Labial", tipo="Barra", tono="Rojo", pvp=23000, mes_lanzamiento=7)

escenarios = [
    ("Caso base", {}),
    ("Canal: Exito", {"canal": "Exito"}),
    ("Canal: Prosalon", {"canal": "Prosalon"}),
    ("Marca: Catrice", {"marca": "Catrice"}),
    ("PVP 15.000 (mas barato)", {"pvp": 15000}),
    ("PVP 30.000 (mas caro)", {"pvp": 30000}),
    ("Tono Nude", {"tono": "Nude"}),
    ("Tipo Liquido", {"tipo": "Liquido"}),
    ("Lanza en enero (S1)", {"mes_lanzamiento": 1}),
    ("Grupo Brillo", {"grupo": "Brillo"}),
]

filas = []
for nombre, cambio in escenarios:
    r = lz.sugerir_lanzamiento(modelo, **{**base, **cambio}, mostrar=False)
    filas.append({
        "Escenario": nombre,
        "P25": r["P25"], "P50": r["P50"], "P75 (sugerido)": r["unidades_sugeridas"],
        "Comparables": r["cohorte_n"],
        "Criterio alcanzado": r["cohorte_nivel"],
        "Revisar": "SI" if r["requiere_revision"] else "",
    })

sensibilidad = pd.DataFrame(filas)
sensibilidad

In [ ]:
# Visualizacion de la sensibilidad
orden = sensibilidad.sort_values("P75 (sugerido)")
fig, ax = plt.subplots(figsize=(11, 6))
colores = ["indianred" if e == "Caso base" else "steelblue" for e in orden["Escenario"]]
ax.barh(orden["Escenario"], orden["P75 (sugerido)"], color=colores)
ax.set_title("Cuanto mueve cada atributo el sugerido\n(caso base en rojo)")
ax.set_xlabel("Unidades sugeridas para los primeros 6 meses (P75)")
for y, v in enumerate(orden["P75 (sugerido)"]):
    ax.text(v, y, f" {v:,.0f}".replace(",", "."), va="center", fontsize=9)
plt.show()

### Lectura de la sensibilidad

**La decisión de canal es la más costosa de equivocar**, muy por encima de cualquier
atributo del producto. Cambiar de Farmatodo a Prosalon cambia el sugerido por un
factor cercano a 10.

Eso tiene una implicación práctica: **la conversación de planeación debería empezar por
"cuánto en cada canal" y solo después por "qué producto"**. Los atributos del producto
afinan la cifra; el canal la define.

## 7. Los lanzamientos pendientes del archivo

Los 385 registros sin una sola venta en 30 meses son, en su mayoría, novedades 2026 que
todavía no han entrado a vender (229 `New's junio 2026` + 123 `New's noviembre 2026`).

En el pipeline de clustering quedaban como "definir comercialmente". Ahora sí reciben
una cifra estimada desde sus atributos.

In [ ]:
sugeridos = lz.puntuar_pendientes(modelo, maestro)
sugeridos.head(15)

In [ ]:
# Los que necesitan validacion comercial antes de pedir
revisar = sugeridos[sugeridos["requiere_revision"]]
print(f"Registros marcados para revision: {len(revisar)} de {len(sugeridos)} "
      f"({utils.porcentaje(len(revisar), len(sugeridos))}%)\n")

print("Por que se marcan: el criterio de cohorte alcanzado fue muy general")
print(revisar["cohorte_nivel"].value_counts().to_string())

print("\nDistribucion del criterio alcanzado en TODOS los sugeridos:")
print(sugeridos["cohorte_nivel"].value_counts().to_string())

### Lectura

El porcentaje marcado para revisión no es un defecto del modelo: **es el modelo siendo
explícito sobre dónde no tiene evidencia suficiente**. Un modelo que entregara 385
cifras con la misma confianza aparente sería menos útil y más peligroso.

La columna `cohorte_nivel` permite priorizar el trabajo del comprador: revisar primero
los que se quedaron en `Canal` o `Canal + Subcategoría`, y confiar en los que llegaron
a `Canal + Grupo + Tipo + Tono`.

## 8. Exportación

In [ ]:
# El Excel final incorpora la hoja 'Sugerido_Lanzamientos'
clusters = pd.read_parquet(cfg.ARCHIVO_CLUSTERS)
from src import evaluation as ev
perfiles = ev.perfilar_clusters(clusters)
clusters, perfiles = ev.asignar_nombres(clusters, perfiles)

ruta = utils.exportar_excel_final(clusters, perfiles, sugeridos)

excel = pd.ExcelFile(cfg.ARCHIVO_EXCEL_FINAL)
print("\nHojas del entregable:\n")
for hoja in excel.sheet_names:
    t = pd.read_excel(cfg.ARCHIVO_EXCEL_FINAL, sheet_name=hoja)
    print(f"  {hoja:<25} {len(t):>6} filas x {t.shape[1]:>3} columnas")

---

## Resumen: cómo se calcula el sugerido de un producto nuevo

**Entrada:** canal, marca, subcategoría, grupo, tipo, tono, PVP y mes de lanzamiento.

**Proceso:**

1. Se calcula `pvp_relativo` = PVP ÷ mediana de PVP de su subcategoría.
2. Tres modelos de Gradient Boosting cuantílico (entrenados en 916 lanzamientos
   históricos reales) predicen P25, P50 y P75 de las unidades a vender en 6 meses.
3. En paralelo, la cohorte de análogos busca productos históricos parecidos en cascada
   y calcula sus cuantiles reales observados.
4. **El sugerido es el P75 del modelo** (nivel de servicio elegido por el negocio: un
   quiebre en el lanzamiento mata el sell-in inicial).
5. Si modelo y cohorte difieren por más del doble, se marca para revisión.

**Salida:** rango P25–P50–P75, sugerido en unidades, ingreso proyectado, criterio de
búsqueda alcanzado, lista de productos comparables con su venta real, y señal de
revisión.

### Limitaciones

1. **Acierta dentro del factor 2 en el 64% de los casos.** Es una estimación con
   incertidumbre real, no una cifra exacta. Por eso se entrega el rango.
2. **No conoce el apoyo comercial.** Plan de medios, exhibición negociada, respaldo
   publicitario: dos productos idénticos con distinto apoyo venden muy distinto y el
   modelo no puede verlo. Es su límite más importante.
3. **Excluye 352 productos censurados** del entrenamiento. Con más años de historia
   ese set crecería y el modelo mejoraría.
4. **Un tono o grupo nunca visto** se predice apoyándose en el resto de los atributos,
   con menos precisión. La señal de revisión avisa cuando eso pasa.

### Siguiente iteración

- Registrar la fecha oficial de lanzamiento de cada producto, en lugar de inferir el
  debut del primer mes con venta. Eso recuperaría parte de los 352 censurados.
- Incorporar el nivel de inversión comercial planeado por producto, que hoy es la
  variable ausente más importante.
- Al cierre de cada semestre, los lanzamientos nuevos se suman al set de entrenamiento
  automáticamente: el modelo mejora solo con volver a correr el pipeline.